In [3]:
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'  # 必须在导入 Unsloth 之前设置！
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [18]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# 1. 加载基座模型（与 DAPT 一致）
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/root/autodl-tmp/DeepSeek-R1-Distill-Qwen-7B",
    max_seq_length=2048,
    load_in_4bit=True,
)

# 2. 加载 DAPT 适配器（40000 步最新版）
model.load_adapter("poetry-sft-adapter")  # 确保路径正确

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:02<00:00, 123.21it/s]


/root/autodl-tmp/DeepSeek-R1-Distill-Qwen-7B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 815.80it/s]


In [22]:
# 测试函数
def ask_poetry_expert(question):
    model_for_infer = FastLanguageModel.for_inference(model)
    messages = [
        {"role": "system", "content": "你是中国古典诗词赏析专家。"},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model_for_infer.generate(**inputs, max_new_tokens=1000, temperature=0.3, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("助手")[-1].strip()

# 测试字词理解
# print(ask_poetry_expert("在诗句“美人清江畔，是夜越吟苦。”中，“美人”是什么意思？"))
# 测试诗句理解
print(ask_poetry_expert("把诗句“溪云初起日沉阁，山雨欲来风满楼”翻译成现代汉语"))

Both `max_new_tokens` (=1000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


你是中国古典诗词赏析专家。<｜User｜>把诗句“溪云初起日沉阁，山雨欲来风满楼”翻译成现代汉语<｜Assistant｜><think>
好的，我现在需要帮用户把诗句“溪云初起日沉阁，山雨欲来风满楼”翻译成现代汉语。首先，我得理解每一句的意思。

第一句“溪云初起日沉阁”中的“溪云”应该是指溪边的云，刚升起，而“日沉阁”可能是指夕阳沉入高阁，整体给人一种宁静的氛围。我可以翻译为“溪边的云刚升起，夕阳沉入高阁。”

接下来是第二句“山雨欲来风满楼”，这里的“山雨欲来”描绘了山间的雨即将落下，而“风满楼”则表现了楼上的风势大。我需要找到合适的词语来表达这种景象，比如“山间云雨渐浓，楼风大作。”

组合起来，整个诗句描绘了溪边云起、夕阳沉入高阁，同时山间云雨渐浓，楼风大作的景象。我需要确保翻译既保留诗意，又通顺自然，让现代读者容易理解。

最后，我会检查一下整个翻译是否流畅，是否准确传达了原诗句的意境。这样用户就能得到一个满意的结果了。
</think>

溪边的云刚升起，夕阳沉入高阁。山间云雨渐浓，楼风大作。


In [13]:
model = FastLanguageModel.for_inference(model)
def generate_continuation(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [17]:
print(generate_continuation("钟山风雨起苍黄，"))

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


钟山风雨起苍黄，白浪翻天浪不扬。
自古英雄多死气，不知谁是汉家郎。
